<div style="background-color: #ffe6e6; padding: 20px; border-radius: 8px; border: 1px solid #ffcccc;">
  <h1 style="color: #cc0000; margin-top: 0;">🚨 O Código Espaguete (O Problema)</h1>
  
  <b>A Dificuldade de Manutenção:</b> Códigos <i>ad hoc</i> com dezenas de passos manuais são impossíveis de sustentar em produção.<br><br>
  <b>O Desperdício de Variáveis:</b> Criar múltiplos DataFrames intermediários (<code>df1</code>, <code>df2</code>, <code>X_train_imputed</code>, <code>X_train_scaled</code>) destrói a memória RAM e a legibilidade.<br><br>
  <b>O Risco Letal:</b> Executar células fora de ordem é a receita perfeita para o <strong>Data Leakage</strong> (Vazamento de Dados).
</div>

## 1. Simulação de Dados Brutos
Gerando alguns dados fictícios com valores ausentes para aplicarmos as transformações.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(42)
n_samples = 1000

df_raw = pd.DataFrame({
    'idade': np.random.normal(35, 10, n_samples),
    'salario': np.random.normal(5000, 2000, n_samples),
    'cidade': np.random.choice(['São Paulo', 'Rio de Janeiro', 'Belo Horizonte'], n_samples),
    'target_comprou': np.random.choice([0, 1], n_samples)
})

# Simulando valores nulos para precisar de imputação
df_raw.loc[np.random.choice(n_samples, 50, replace=False), 'idade'] = np.nan
df_raw.loc[np.random.choice(n_samples, 30, replace=False), 'salario'] = np.nan

df_raw.head()

## 2. O pesadelo do Cientista de Dados

Preste atenção na quantidade de variáveis vazando e espalhadas ao longo das transformações. **Se você receber um conjunto de dados novo agora**, vai ter que replicar a ordem exata das criações de objetos para não gerar erros e vazar dados.

In [ ]:
df1 = df_raw.copy()
df2 = df1.dropna(subset=['target_comprou'])

X_raw = df2.drop('target_comprou', axis=1)
y = df2['target_comprou']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# --- SEPARANDO VARIÁVEIS NUMÉRICAS E CATEGÓRICAS ---
X_train_num = X_train_raw[['idade', 'salario']].copy()
X_test_num = X_test_raw[['idade', 'salario']].copy()

X_train_cat = X_train_raw[['cidade']].copy()
X_test_cat = X_test_raw[['cidade']].copy()

# --- IMPUTAÇÃO DE VALORES NULOS ---
imputer = SimpleImputer(strategy='median')

X_train_imputed = imputer.fit_transform(X_train_num)
X_test_imputed = imputer.transform(X_test_num)

# Transformar de volta para DataFrame 
X_train_num_df = pd.DataFrame(X_train_imputed, columns=['idade', 'salario'], index=X_train_raw.index)
X_test_num_df = pd.DataFrame(X_test_imputed, columns=['idade', 'salario'], index=X_test_raw.index)

# --- ONE HOT ENCODING ---
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_encoded = encoder.fit_transform(X_train_cat)
X_test_encoded = encoder.transform(X_test_cat)

cat_columns = encoder.get_feature_names_out(['cidade'])

# Transformar de volta para DataFrame
X_train_cat_df = pd.DataFrame(X_train_encoded, columns=cat_columns, index=X_train_raw.index)
X_test_cat_df = pd.DataFrame(X_test_encoded, columns=cat_columns, index=X_test_raw.index)

# --- REUNINDO O DADO ---
X_train_joined = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_joined = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

# --- SCALER ---
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_joined)
X_test_scaled = scaler.transform(X_test_joined)

# Transformar de volta para DataFrame
X_train_FINAL = pd.DataFrame(X_train_scaled, columns=X_train_joined.columns, index=X_train_raw.index)
X_test_FINAL = pd.DataFrame(X_test_scaled, columns=X_test_joined.columns, index=X_test_raw.index)

X_train_FINAL.head()

### 🗣️ Notas do Orador (Talking Points):
- <i>"Olhem para este código na tela. Imputamos, aplicamos o One-Hot, depois o Scaler... São quase 50 linhas e mais de 10 variáveis diferentes."</i>
- <i>"Eu faço uma pergunta a vocês: Se o chefe de vocês mandar um arquivo novo com dados atualizados de hoje, vocês vão abrir o Jupyter, apertar 'Shift+Enter' 50 vezes na ordem exata e rezar para não dar erro?"</i>
- <i>"Em produção, isso é inaceitável. Precisamos de um objeto único que receba o dado sujo de um lado e cuspa o dado limpo do outro."</i>